# 02 - Stochastic Cycle Model — Brazil IPCA Inflation

## Introduction

The **stochastic cycle** model (Harvey, 1989) represents cyclical dynamics as a
rotation in the complex plane with damping:

$$
\begin{bmatrix} \psi_t \\ \psi^*_t \end{bmatrix} =
\rho \begin{bmatrix} \cos \lambda_c & \sin \lambda_c \\ -\sin \lambda_c & \cos \lambda_c \end{bmatrix}
\begin{bmatrix} \psi_{t-1} \\ \psi^*_{t-1} \end{bmatrix} +
\begin{bmatrix} \kappa_t \\ \kappa^*_t \end{bmatrix}
$$

where:
- $0 < \rho < 1$ is the **damping factor** (persistence of the cycle)
- $\lambda_c = 2\pi / T$ is the **cycle frequency** ($T$ = period in data units)
- $\kappa_t, \kappa^*_t \sim N(0, \sigma^2_\kappa)$ are the cycle disturbances

The **amplitude** of the cycle at time $t$ is $A_t = \sqrt{\psi_t^2 + \psi^{*2}_t}$
and the **phase** is $\phi_t = \arctan(\psi^*_t / \psi_t)$.

We apply this model to **Brazil monthly IPCA inflation** (2000-2023) to extract
the cyclical component of Brazilian inflation dynamics.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import signal, optimize

from kalmanbox import UnobservedComponents
from kalmanbox.models.cycle import CycleModel
from kalmanbox.datasets import load_dataset
from kalmanbox.filters.kalman import KalmanFilter
from kalmanbox.smoothers.rts import RTSSmoother

import statsmodels.api as sm
from statsmodels.tsa.filters.hp_filter import hpfilter
from statsmodels.tsa.filters.cf_filter import cffilter

import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.figsize': (12, 5),
    'axes.grid': True,
    'grid.alpha': 0.3,
})

print('Imports OK')

In [ ]:
# Load Brazil IPCA inflation data
df = load_dataset('brazil_ipca')
dates = pd.to_datetime(df['date'])
y = df['ipca'].to_numpy(dtype=np.float64)

print(f'Shape: {df.shape}')
print(f'Period: {dates.iloc[0].strftime("%Y-%m")} to {dates.iloc[-1].strftime("%Y-%m")}')
print(f'\nDescriptive statistics:')
print(f'  Mean: {y.mean():.3f}% per month')
print(f'  Std:  {y.std():.3f}%')
print(f'  Min:  {y.min():.3f}%, Max: {y.max():.3f}%')
print(f'  Annualized mean: {((1 + y.mean()/100)**12 - 1)*100:.1f}%')
df.head()

In [ ]:
# Spectral analysis (periodogram) to identify dominant frequencies
# Remove mean before spectral analysis
y_centered = y - y.mean()

freqs, psd = signal.periodogram(y_centered, fs=12)  # fs=12 months/year

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Time series plot
axes[0].plot(dates, y, 'k-', linewidth=0.7)
axes[0].axhline(y.mean(), color='red', linestyle='--', alpha=0.5, label=f'Mean = {y.mean():.3f}%')
axes[0].set_xlabel('Date')
axes[0].set_ylabel('Monthly IPCA (%)')
axes[0].set_title('Brazil Monthly IPCA Inflation')
axes[0].legend()

# Periodogram
# Convert frequency (cycles/year) to period (years)
period_years = np.zeros_like(freqs)
mask = freqs > 0
period_years[mask] = 1.0 / freqs[mask]

axes[1].semilogy(freqs[mask], psd[mask], 'b-', linewidth=0.7)
# Mark peaks
peak_idx = signal.argrelmax(psd[mask], order=3)[0]
if len(peak_idx) > 0:
    top_peaks = peak_idx[np.argsort(psd[mask][peak_idx])[-5:]]
    for pi in top_peaks:
        if period_years[mask][pi] > 0.5:  # Only label periods > 6 months
            axes[1].axvline(freqs[mask][pi], color='red', alpha=0.3, linewidth=0.8)
            axes[1].annotate(f'{period_years[mask][pi]:.1f}y',
                           xy=(freqs[mask][pi], psd[mask][pi]),
                           fontsize=8, color='red')

axes[1].set_xlabel('Frequency (cycles/year)')
axes[1].set_ylabel('Power Spectral Density')
axes[1].set_title('Periodogram of IPCA')

plt.tight_layout()
plt.show()

# Report dominant periods
sorted_peaks = np.argsort(psd[1:])[::-1] + 1
print('Top 5 spectral peaks:')
for i in range(min(5, len(sorted_peaks))):
    idx = sorted_peaks[i]
    if freqs[idx] > 0:
        print(f'  Period = {1/freqs[idx]:.1f} years (freq = {freqs[idx]:.3f} cycles/year), PSD = {psd[idx]:.4f}')

In [ ]:
# Model: Local Level + Stochastic Cycle
# y_t = mu_t + psi_t + eps_t
# mu_t = mu_{t-1} + eta_t (random walk level)
# psi_t = damped rotation cycle

ucm = UnobservedComponents(
    y,
    level=True,
    trend='none',
    cycle=True,
)

print(f'Model: Local Level + Stochastic Cycle')
print(f'Number of observations: {len(y)}')
print(f'Number of states: {ucm._k_states}')
print(f'Parameters: {ucm.param_names}')
print(f'State layout: {ucm._layout}')

In [ ]:
# MLE estimation using differential evolution (global optimizer)
# Params: [sigma2_obs, sigma2_level, rho, lambda_c, sigma2_cycle]
bounds_de = [
    (-8, 3),      # log(sigma2_obs)
    (-10, 3),     # log(sigma2_level)
    (-1.5, 3.0),  # arctanh(2*rho-1) -> rho in ~(0.1, 0.995)
    (-3.5, 0.5),  # logit(lambda_c/pi) -> wide frequency range
    (-10, 3),     # log(sigma2_cycle)
]

def neg_loglike(unc):
    try:
        constrained = ucm.transform_params(unc)
        return -ucm.loglike(constrained)
    except Exception:
        return 1e10

de_result = optimize.differential_evolution(
    neg_loglike, bounds_de, seed=42, maxiter=300, tol=1e-10,
    popsize=20, mutation=(0.5, 1.5), recombination=0.9
)

# Refine with L-BFGS-B
refined = optimize.minimize(
    neg_loglike, de_result.x, method='L-BFGS-B', bounds=bounds_de,
    options={'maxiter': 2000, 'ftol': 1e-12}
)
if refined.fun > de_result.fun:
    refined = de_result

optimal_params = ucm.transform_params(refined.x)

print('Estimated parameters (MLE):')
print('=' * 50)
for name, val in zip(ucm.param_names, optimal_params):
    print(f'  {name:20s} = {val:.6f}')
print(f'\nLog-likelihood: {-refined.fun:.4f}')

rho_hat = optimal_params[ucm.param_names.index('rho')]
lambda_c_hat = optimal_params[ucm.param_names.index('lambda_c')]
period_months = 2 * np.pi / lambda_c_hat
period_years = period_months / 12

print(f'\nCycle frequency (lambda_c): {lambda_c_hat:.4f} rad/period')
print(f'Estimated period: {period_months:.1f} months = {period_years:.1f} years')
print(f'Damping factor (rho): {rho_hat:.4f}')

In [ ]:
# Periodicity analysis: estimated vs theoretical
print('Cycle Periodicity Analysis')
print('=' * 50)
print(f'Estimated frequency: lambda_c = {lambda_c_hat:.4f}')
print(f'Estimated period: T = 2*pi/lambda_c = {period_months:.1f} months')
print(f'                                     = {period_years:.1f} years')
print(f'Damping factor: rho = {rho_hat:.4f}')
if rho_hat > 0 and rho_hat < 1:
    half_life = -np.log(2) / np.log(rho_hat)
    print(f'Half-life: {half_life:.1f} months = {half_life/12:.1f} years')
print(f'\nVariance decomposition:')
sig2_obs = optimal_params[ucm.param_names.index('sigma2_obs')]
sig2_level = optimal_params[ucm.param_names.index('sigma2_level')]
sig2_cycle = optimal_params[ucm.param_names.index('sigma2_cycle')]
total_var = sig2_obs + sig2_level + sig2_cycle
print(f'  sigma2_obs / total   = {sig2_obs/total_var*100:.1f}% (irregular)')
print(f'  sigma2_level / total = {sig2_level/total_var*100:.1f}% (level shifts)')
print(f'  sigma2_cycle / total = {sig2_cycle/total_var*100:.1f}% (cyclical)')

In [ ]:
# Extract smoothed cycle component
ssm = ucm._build_ssm(optimal_params)
kf = KalmanFilter()
smoother = RTSSmoother()
fo = kf.filter(y.reshape(-1, 1), ssm)
so = smoother.smooth(fo, ssm)

layout = ucm._layout
smoothed = so.smoothed_state

level = smoothed[:, layout['level']['start']]
cycle_psi = smoothed[:, layout['cycle']['start']]
cycle_psi_star = smoothed[:, layout['cycle']['start'] + 1]

# Plot: original data with extracted cycle
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# Observed + level
axes[0].plot(dates, y, 'k-', linewidth=0.6, alpha=0.6, label='IPCA observed')
axes[0].plot(dates, level, 'b-', linewidth=1.5, label='Level (smoothed)')
axes[0].set_ylabel('Monthly Inflation (%)')
axes[0].set_title('IPCA Decomposition: Level + Cycle')
axes[0].legend()

# Extracted cycle
axes[1].plot(dates, cycle_psi, 'r-', linewidth=1.2, label='Cycle $\\psi_t$')
axes[1].fill_between(dates, cycle_psi, 0, alpha=0.15, color='red')
axes[1].axhline(0, color='gray', linewidth=0.5)
axes[1].set_ylabel('Cycle (%)')
axes[1].set_title(f'Extracted Stochastic Cycle (Period = {period_months:.0f} months = {period_years:.1f} years)')
axes[1].legend()

# Residuals
residuals = y - level - cycle_psi
axes[2].plot(dates, residuals, 'gray', linewidth=0.6)
axes[2].axhline(0, color='black', linewidth=0.5)
axes[2].set_ylabel('Irregular (%)')
axes[2].set_title('Irregular Component')
axes[2].set_xlabel('Date')

plt.tight_layout()
plt.show()

In [ ]:
# Cycle amplitude and phase evolution over time
amplitude = np.sqrt(cycle_psi**2 + cycle_psi_star**2)
phase = np.arctan2(cycle_psi_star, cycle_psi)

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Amplitude
axes[0].plot(dates, amplitude, 'm-', linewidth=1.2)
axes[0].fill_between(dates, 0, amplitude, alpha=0.15, color='purple')
axes[0].set_ylabel('Amplitude (%)')
axes[0].set_title('Cycle Amplitude Over Time')

# Phase
axes[1].plot(dates, phase, 'c-', linewidth=0.8)
axes[1].set_ylabel('Phase (radians)')
axes[1].set_title('Cycle Phase Over Time')
axes[1].set_xlabel('Date')

plt.tight_layout()
plt.show()

print(f'Amplitude statistics:')
print(f'  Mean: {amplitude.mean():.4f}%')
print(f'  Max:  {amplitude.max():.4f}% at {dates.iloc[np.argmax(amplitude)].strftime("%Y-%m")}')
print(f'  Min:  {amplitude.min():.4f}% at {dates.iloc[np.argmin(amplitude)].strftime("%Y-%m")}')

In [ ]:
# Comparison with classical filters: HP and Christiano-Fitzgerald (CF)

# HP filter (lambda=14400 for monthly data)
cycle_hp, trend_hp = hpfilter(y, lamb=14400)

# CF band-pass filter (18-96 months = 1.5-8 years)
cycle_cf, trend_cf = cffilter(y, low=18, high=96)

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# Kalman cycle
axes[0].plot(dates, cycle_psi, 'r-', linewidth=1.2, label='Kalman stochastic cycle')
axes[0].axhline(0, color='gray', linewidth=0.5)
axes[0].set_ylabel('Cycle (%)')
axes[0].set_title('Stochastic Cycle (kalmanbox UCM)')
axes[0].legend()

# HP cycle
axes[1].plot(dates, cycle_hp, 'g-', linewidth=1.2, label='HP filter cycle ($\\lambda=14400$)')
axes[1].axhline(0, color='gray', linewidth=0.5)
axes[1].set_ylabel('Cycle (%)')
axes[1].set_title('HP Filter Cycle')
axes[1].legend()

# CF cycle
n_cf = len(cycle_cf)
axes[2].plot(dates[:n_cf], cycle_cf, 'b-', linewidth=1.2,
             label='CF band-pass (18-96 months)')
axes[2].axhline(0, color='gray', linewidth=0.5)
axes[2].set_ylabel('Cycle (%)')
axes[2].set_title('Christiano-Fitzgerald Band-Pass Filter')
axes[2].set_xlabel('Date')
axes[2].legend()

plt.tight_layout()
plt.show()

# Correlation between cycle estimates
n_min = min(len(cycle_psi), len(cycle_hp), len(cycle_cf))
corr_hp = np.corrcoef(cycle_psi[:n_min], cycle_hp[:n_min])[0, 1]
corr_cf = np.corrcoef(cycle_psi[:n_min], cycle_cf[:n_min])[0, 1]
corr_hp_cf = np.corrcoef(cycle_hp[:n_min], cycle_cf[:n_min])[0, 1]

print('Correlation between cycle estimates:')
print(f'  Kalman vs HP:  {corr_hp:.3f}')
print(f'  Kalman vs CF:  {corr_cf:.3f}')
print(f'  HP vs CF:      {corr_hp_cf:.3f}')

In [ ]:
# Comparison with statsmodels UnobservedComponents
sm_model = sm.tsa.UnobservedComponents(
    y,
    level='local level',
    cycle=True,
    stochastic_cycle=True,
    damped_cycle=True,
)
sm_results = sm_model.fit(disp=False)

sm_freq = sm_results.params[sm_results.param_names.index('frequency.cycle')]
sm_rho = sm_results.params[sm_results.param_names.index('damping.cycle')]
sm_period = 2 * np.pi / sm_freq

print('statsmodels UCM Results:')
print(sm_results.summary())

# Comparison table
comparison = pd.DataFrame({
    'Metric': [
        'Log-Likelihood',
        'Cycle Period (months)',
        'Cycle Period (years)',
        'Damping (rho)',
        'sigma2_obs',
        'sigma2_cycle',
    ],
    'kalmanbox': [
        f'{-refined.fun:.2f}',
        f'{period_months:.1f}',
        f'{period_years:.1f}',
        f'{rho_hat:.4f}',
        f'{sig2_obs:.6f}',
        f'{sig2_cycle:.6f}',
    ],
    'statsmodels': [
        f'{sm_results.llf:.2f}',
        f'{sm_period:.1f}',
        f'{sm_period/12:.1f}',
        f'{sm_rho:.4f}',
        f'{sm_results.params[sm_results.param_names.index("sigma2.irregular")]:.6f}',
        f'{sm_results.params[sm_results.param_names.index("sigma2.cycle")]:.6f}',
    ],
})

print('\nComparison: kalmanbox vs statsmodels')
print('=' * 70)
print(comparison.to_string(index=False))

## Conclusions

1. **Spectral Analysis**: The periodogram reveals the dominant frequencies in
   Brazilian inflation data, guiding the choice of cycle period bounds.

2. **Stochastic Cycle Extraction**: The UCM with local level + stochastic cycle
   successfully extracts a cyclical component from IPCA inflation. The estimated
   period and damping factor characterize the inflation dynamics.

3. **Amplitude Dynamics**: The cycle amplitude varies over time, reflecting
   periods of more or less pronounced cyclical fluctuations in inflation.

4. **Filter Comparison**: The Kalman-based stochastic cycle differs from
   mechanical filters (HP, CF) because:
   - It is **model-based** with a probabilistic interpretation
   - It estimates the cycle frequency and damping from the data
   - It provides **optimal** signal extraction under the model assumptions
   - HP and CF filters are symmetric (use future data) while the Kalman filter
     can also produce one-sided (real-time) estimates

5. **Comparison with statsmodels**: Both implementations identify cyclical
   dynamics, though they may find different local optima due to the
   multimodal nature of the cycle frequency likelihood surface.

### References
- Harvey, A.C. (1989). *Forecasting, Structural Time Series Models and the Kalman Filter*.
- Hodrick, R.J. & Prescott, E.C. (1997). Postwar U.S. Business Cycles. *JMCB*.
- Christiano, L.J. & Fitzgerald, T.J. (2003). The Band Pass Filter. *IER*.